# BirdCLEF v2 training (Kaggle)

Goal: **beat v1 val macro ROC-AUC = 0.8529** with a simple, junior-style recipe:

1. Same model: EfficientNet-B0 + SED + attention  
2. SpecAugment  
3. BCE `pos_weight`  
4. Balanced sampling by `primary_label`  
5. 15 epochs, slightly lower LR  
6. Extra metrics: PR-AUC, F1, precision, recall  

**How to use on Kaggle**
1. Create a new Notebook  
2. Add competition data (BirdCLEF+ 2025 or your train CSV + audio)  
3. Add this repo as a **Dataset** (upload zip of the BirdCLEF folder) **or** copy the `src/` files into the notebook environment  
4. Edit the PATHS cell  
5. Run all — watch the training yourself; download `model_best.pth` + `metrics.json` when done  

Do **not** overwrite v1 `results/v1/`. After a good run, freeze as `results/v2/...` on your PC.

## 0) Install / imports

In [ ]:
import os
import sys
import json
import time
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.cuda.amp import GradScaler, autocast
from torch.utils.data import DataLoader, WeightedRandomSampler
from sklearn.model_selection import train_test_split
from tqdm.auto import tqdm

print('torch', torch.__version__)
print('cuda available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('gpu:', torch.cuda.get_device_name(0))

## 1) PATHS — edit these for your Kaggle inputs

Typical Kaggle layout:
- Competition: `/kaggle/input/birdclef-2025/` (name may differ)
- Your code dataset: `/kaggle/input/birdclef-code/` (whatever you named the upload)

In [ ]:
# ============ EDIT ME ============
CODE_DIR = Path('/kaggle/input/birdclef-code')  # folder that contains src/, config_v2.json
# If you uploaded the whole BirdCLEF repo zip, it might be nested:
# CODE_DIR = Path('/kaggle/input/birdclef-code/BirdCLEF')

DATA_ROOT = Path('/kaggle/input/birdclef-2025')  # competition dataset name — change if needed
TRAIN_CSV = DATA_ROOT / 'train.csv'             # change if your CSV path differs
AUDIO_DIR = DATA_ROOT / 'train_audio'           # raw audio (used if no mel cache)

# Optional: precomputed mels (much faster). Leave as None to compute on the fly (slow).
MEL_DIR = None  # e.g. Path('/kaggle/input/birdclef-mels')

OUT_DIR = Path('/kaggle/working/v2_run')
OUT_DIR.mkdir(parents=True, exist_ok=True)

# quick training test on a small subset? Set to None for full data.
MAX_ROWS = None  # e.g. 2000 for a smoke run on Kaggle

print('CODE_DIR exists:', CODE_DIR.exists(), CODE_DIR)
print('TRAIN_CSV exists:', TRAIN_CSV.exists(), TRAIN_CSV)
print('AUDIO_DIR exists:', AUDIO_DIR.exists() if AUDIO_DIR else None, AUDIO_DIR)
print('MEL_DIR:', MEL_DIR)

In [ ]:
# Put repo on path so `import src...` works
if CODE_DIR.exists():
    sys.path.insert(0, str(CODE_DIR))
else:
    # fallback: notebook copied next to src/
    sys.path.insert(0, str(Path.cwd()))
    sys.path.insert(0, str(Path.cwd().parent))

from src.dataset import (
    BirdCLEFDataset,
    build_target,
    labels_from_metadata,
    load_label_maps,
)
from src.metrics import compute_metrics, per_class_report, print_metrics
from src.model import BirdCLEFSED
from src.utils import load_config, set_seed, save_json

print('imports OK')

## 2) Load config v2

In [ ]:
cfg_path = CODE_DIR / 'config_v2.json'
if not cfg_path.exists():
    cfg_path = Path('config_v2.json')
cfg = load_config(cfg_path)
print(json.dumps(cfg, indent=2))

## 3) Data + split (same style as v1: stratified 80/20 on primary_label)

In [ ]:
df = pd.read_csv(TRAIN_CSV)
if MAX_ROWS is not None:
    df = df.sample(n=min(MAX_ROWS, len(df)), random_state=int(cfg.get('SEED', 42))).reset_index(drop=True)
    print('Using subset rows:', len(df))

print(df.columns.tolist())
print('rows:', len(df))
print(df['primary_label'].value_counts().head())

# Prefer label maps from the uploaded code (same 206 classes as v1)
label_root = CODE_DIR if (CODE_DIR / 'label2id.json').exists() else Path('.')
if (label_root / 'label2id.json').exists():
    classes, label2id = load_label_maps(label_root)
else:
    classes, label2id = labels_from_metadata(df)

num_classes = len(label2id)
cfg['NUM_CLASSES'] = num_classes
print('num_classes:', num_classes)

seed = int(cfg.get('SEED', 42))
val_ratio = float(cfg.get('VAL_RATIO', 0.2))
train_df, val_df = train_test_split(
    df,
    test_size=val_ratio,
    random_state=seed,
    stratify=df['primary_label'] if 'primary_label' in df.columns else None,
)
train_df = train_df.reset_index(drop=True)
val_df = val_df.reset_index(drop=True)
print('train', len(train_df), 'val', len(val_df))

## 4) Datasets / loaders (SpecAugment + balanced sampler)

In [ ]:
set_seed(seed)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

audio_dir = str(AUDIO_DIR) if AUDIO_DIR is not None and Path(AUDIO_DIR).exists() else None
mel_dir = str(MEL_DIR) if MEL_DIR is not None else None
if audio_dir is None and mel_dir is None:
    raise RuntimeError('Need AUDIO_DIR or MEL_DIR — edit the PATHS cell')

train_ds = BirdCLEFDataset(train_df, label2id, cfg, audio_dir=audio_dir, mel_dir=mel_dir, train=True)
val_ds = BirdCLEFDataset(val_df, label2id, cfg, audio_dir=audio_dir, mel_dir=mel_dir, train=False)

# quick shape check
x0, y0 = train_ds[0]
print('sample x', tuple(x0.shape), 'y sum', float(y0.sum()))

batch_size = int(cfg.get('BATCH_SIZE', 16))

# balanced sampler by primary_label
counts = train_df['primary_label'].astype(str).value_counts()
weights = train_df['primary_label'].astype(str).map(lambda x: 1.0 / float(counts[x])).to_numpy()
sampler = WeightedRandomSampler(
    weights=torch.as_tensor(weights, dtype=torch.double),
    num_samples=len(weights),
    replacement=True,
)

train_loader = DataLoader(
    train_ds,
    batch_size=batch_size,
    sampler=sampler,
    num_workers=2,
    pin_memory=device.type == 'cuda',
    drop_last=True,
)
val_loader = DataLoader(
    val_ds,
    batch_size=batch_size * 2,
    shuffle=False,
    num_workers=2,
    pin_memory=device.type == 'cuda',
)
print('loaders OK', len(train_loader), len(val_loader))

## 5) Model + pos_weight loss

In [ ]:
def compute_pos_weight(train_df, label2id, max_weight=50.0):
    n = len(train_df)
    num_classes = len(label2id)
    counts = np.zeros(num_classes, dtype=np.float64)
    for _, row in train_df.iterrows():
        counts += build_target(row, label2id, num_classes)
    pos = np.clip(counts, 1.0, None)
    neg = np.clip(n - counts, 0.0, None)
    w = np.clip(neg / pos, 1.0, max_weight).astype(np.float32)
    print(f'pos_weight min={w.min():.2f} max={w.max():.2f} median={np.median(w):.2f}')
    return torch.tensor(w, dtype=torch.float32)

epochs = int(cfg.get('EPOCHS', 15))
lr = float(cfg.get('LR', 8e-4))
weight_decay = float(cfg.get('WEIGHT_DECAY', 1e-5))
dropout = float(cfg.get('DROPOUT', 0.35))
threshold = float(cfg.get('THRESHOLD', 0.5))
backbone = str(cfg.get('BACKBONE', 'efficientnet_b0'))

model = BirdCLEFSED(
    num_classes=num_classes,
    backbone_name=backbone,
    pretrained=True,
    dropout=dropout,
).to(device)

pos_weight = compute_pos_weight(
    train_df, label2id, max_weight=float(cfg.get('POS_WEIGHT_MAX', 50.0))
).to(device)
criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=weight_decay)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs)
scaler = GradScaler(enabled=device.type == 'cuda')

print(model.feature_dim, 'params', sum(p.numel() for p in model.parameters())/1e6, 'M')

## 6) Train (you watch this cell — long on full data)

In [ ]:
@torch.no_grad()
def evaluate_full(model, loader, device, criterion, threshold=0.5):
    model.eval()
    losses, ys, ps = [], [], []
    for x, y in loader:
        x = x.to(device, non_blocking=True)
        y = y.to(device, non_blocking=True)
        with autocast(enabled=device.type == 'cuda'):
            logits, _ = model(x)
            loss = criterion(logits, y)
        losses.append(loss.item())
        ys.append(y.cpu().numpy())
        ps.append(torch.sigmoid(logits).float().cpu().numpy())
    y_true = np.concatenate(ys, 0)
    y_prob = np.concatenate(ps, 0)
    m = compute_metrics(y_true, y_prob, threshold=threshold)
    m['val_loss'] = float(np.mean(losses))
    m['_y_true'] = y_true
    m['_y_prob'] = y_prob
    return m

history = []
best_auc = -1.0
best_metrics = None
best_path = OUT_DIR / 'model_best.pth'

print('v1 baseline to beat: 0.8529')
print(f'train {len(train_ds)} val {len(val_ds)} epochs {epochs} lr {lr} device {device}')

for epoch in range(1, epochs + 1):
    model.train()
    t0 = time.time()
    running = 0.0
    pbar = tqdm(train_loader, desc=f'Epoch {epoch}/{epochs}')
    for x, y in pbar:
        x = x.to(device, non_blocking=True)
        y = y.to(device, non_blocking=True)
        optimizer.zero_grad(set_to_none=True)
        with autocast(enabled=device.type == 'cuda'):
            logits, _ = model(x)
            loss = criterion(logits, y)
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()
        running += loss.item()
        pbar.set_postfix(loss=f'{loss.item():.4f}')

    scheduler.step()
    train_loss = running / max(1, len(train_loader))
    val_m = evaluate_full(model, val_loader, device, criterion, threshold=threshold)
    y_true = val_m.pop('_y_true')
    y_prob = val_m.pop('_y_prob')
    elapsed = time.time() - t0

    row = {'epoch': epoch, 'train_loss': train_loss, 'seconds': elapsed, 'lr': scheduler.get_last_lr()[0], **val_m}
    history.append(row)
    print(
        f"Epoch {epoch:02d} | train_loss={train_loss:.4f} | val_loss={val_m['val_loss']:.4f} | "
        f"AUC={val_m['macro_roc_auc']:.4f} | macro_f1={val_m['macro_f1']:.4f} | "
        f"micro_f1={val_m['micro_f1']:.4f} | {elapsed:.1f}s"
    )

    if val_m['macro_roc_auc'] > best_auc:
        best_auc = float(val_m['macro_roc_auc'])
        best_metrics = dict(val_m)
        torch.save(model.state_dict(), best_path)
        torch.save(model.state_dict(), OUT_DIR / 'birdclef_v2_best_model.pth')
        pd.DataFrame(per_class_report(y_true, y_prob, class_names=classes, threshold=threshold)).to_csv(
            OUT_DIR / 'per_class_metrics.csv', index=False
        )
        print(f'  -> new best AUC={best_auc:.4f}')

torch.save(model.state_dict(), OUT_DIR / 'model_last.pth')
print_metrics(best_metrics, title=f'Best val (AUC={best_auc:.4f})')
print(f'v1 baseline 0.8529 | v2 best {best_auc:.4f} | delta {best_auc - 0.8529:+.4f}')

summary = {
    'version': 'v2',
    'best_val_auc': best_auc,
    'v1_baseline_auc': 0.8529,
    'delta_vs_v1': best_auc - 0.8529,
    'best_metrics': best_metrics,
    'history': history,
    'config': cfg,
    'train_size': len(train_ds),
    'val_size': len(val_ds),
    'num_classes': num_classes,
}
save_json(summary, OUT_DIR / 'metrics.json')
save_json(cfg, OUT_DIR / 'config_used.json')
print('saved to', OUT_DIR)
print('Download: model_best.pth, metrics.json, per_class_metrics.csv')

## 7) After training (on your PC)

1. Download `model_best.pth`, `metrics.json`, `per_class_metrics.csv`  
2. Put them under something like `results/v2/`  
3. Compare to `results/v1/` (NOTES.md + training_summary.json)  
4. Only promote root `model.pth` if v2 really beats 0.8529